# 🔄 Dataset Augmentation Generator

Generates dataset variants for the Single Model architecture to test robustness across different conditions.

### Generated Variants:
- **D2a**: CLAHE + Mild Brightness/Contrast (Test impact of illumination normalization)
- **D2b**: D2a + Color Jitter (minimal) (Test model sensitivity to color variation)
- **D2c (full)**: D2b + Geometric (Flip/Rotation) (Test orientation invariance)

In [ ]:
import os
import cv2
import ast
import shutil
from pathlib import Path
import numpy as np
import albumentations as A
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## ⚙️ Configuration
Set the input directory of the original YOLO dataset and the output base directory.

In [ ]:
INPUT_DATASET_DIR = "../../datasets/batch4"
OUTPUT_BASE_DIR = "../../datasets/variants"

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

ensure_dir(OUTPUT_BASE_DIR)

## 🧪 Albumentations Pipelines

In [ ]:
# Variant D2a: CLAHE + Mild Brightness/Contrast
pipline_d2a = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0)
], keypoint_params=A.KeypointParams(format='xy'))

# Variant D2b: D2a + Color Jitter (minimal)
pipline_d2b = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
    A.ColorJitter(brightness=0, contrast=0, saturation=0.1, hue=0.05, p=1.0)
], keypoint_params=A.KeypointParams(format='xy'))

# Variant D2c: D2b + Geometric (Flip/Rotation)
pipline_d2c = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
    A.ColorJitter(brightness=0, contrast=0, saturation=0.1, hue=0.05, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.SafeRotate(limit=45, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7)
], keypoint_params=A.KeypointParams(format='xy'))

PIPELINES = {
    'D2a': pipline_d2a,
    'D2b': pipline_d2b,
    'D2c': pipline_d2c
}

## 🛠️ Processing Logic
Read YOLO format instance segmentation annotations, handle pixel conversion for Albumentations keypoints, and write back normalized coordinates.

In [ ]:
def process_image_and_label(img_path, label_path, out_img_path, out_label_path, transform):
    """
    Reads an image and its corresponding YOLO segmentation label,
    applies the transform, and saves the augmented outputs.
    """
    # Read image
    image = cv2.imread(str(img_path))
    if image is None:
        return False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    # Read annotations (class_id x1 y1 x2 y2...)
    classes = []
    keypoints = []
    keypoint_group_sizes = [] # Track how many points belong to which class/polygon

    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = parts[0]
                    pts = [float(p) for p in parts[1:]]
                    # convert normalized to pixel coordinates
                    pts_xy = [(pts[i] * w, pts[i+1] * h) for i in range(0, len(pts)-1, 2)]
                    classes.append(class_id)
                    keypoints.extend(pts_xy)
                    keypoint_group_sizes.append(len(pts_xy))

    # Apply transform
    if len(keypoints) > 0:
        try:
            transformed = transform(image=image, keypoints=keypoints)
            trans_image = transformed['image']
            trans_keypoints = transformed['keypoints']
        except Exception as e:
            print(f"Transformation failed on {img_path}: {e}")
            trans_image = image
            trans_keypoints = keypoints
    else:
        transformed = transform(image=image, keypoints=[])
        trans_image = transformed['image']
        trans_keypoints = []

    # Save transformed image
    trans_image_bgr = cv2.cvtColor(trans_image, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(out_img_path), trans_image_bgr)

    # Save transformed labels
    if len(trans_keypoints) > 0:
        new_h, new_w = trans_image.shape[:2]
        with open(out_label_path, 'w') as f:
            current_idx = 0
            for class_id, size in zip(classes, keypoint_group_sizes):
                poly_pts = trans_keypoints[current_idx:current_idx+size]
                current_idx += size
                
                # Normalize back to 0-1 range
                norm_pts = []
                for (px, py) in poly_pts:
                    px_norm = max(0.0, min(1.0, px / new_w))
                    py_norm = max(0.0, min(1.0, py / new_h))
                    norm_pts.extend([f"{px_norm:.6f}", f"{py_norm:.6f}"])
                
                if len(norm_pts) > 0:
                    f.write(f"{class_id} {" ".join(norm_pts)}\n")

    return True

## 🚀 Generation Pipeline

In [ ]:
def generate_variants():
    input_path = Path(INPUT_DATASET_DIR)
    
    if not input_path.exists():
        print(f"Warning: Dataset path {INPUT_DATASET_DIR} does not exist. Update the INPUT_DATASET_DIR variable.")
        return

    splits = ['train', 'valid', 'test']
    
    for variant_name, pipeline in PIPELINES.items():
        print(f"\n==== Generating Variant: {variant_name} ====")
        variant_dir = Path(OUTPUT_BASE_DIR) / f"{input_path.name}_{variant_name}"
        
        # Copy data.yaml if exists
        yaml_path = input_path / "data.yaml"
        if yaml_path.exists():
            ensure_dir(variant_dir)
            shutil.copy(yaml_path, variant_dir / "data.yaml")

        for split in splits:
            img_in_dir = input_path / split / "images"
            lbl_in_dir = input_path / split / "labels"
            
            if not img_in_dir.exists():
                continue
                
            img_out_dir = variant_dir / split / "images"
            lbl_out_dir = variant_dir / split / "labels"
            ensure_dir(img_out_dir)
            ensure_dir(lbl_out_dir)

            images = list(img_in_dir.glob("*.png")) + list(img_in_dir.glob("*.jpg"))
            
            print(f"Processing {split} split...")
            for img_path in tqdm(images):
                label_path = lbl_in_dir / (img_path.stem + ".txt")
                out_img = img_out_dir / img_path.name
                out_label = lbl_out_dir / (img_path.stem + ".txt")
                
                if split == 'train':
                    # Apply augmentations ONLY to the training set
                    process_image_and_label(img_path, label_path, out_img, out_label, pipeline)
                else:
                    # For validation and test sets, copy unaltered images and labels to avoid data leakage
                    shutil.copy(img_path, out_img)
                    if label_path.exists():
                        shutil.copy(label_path, out_label)

    print("\n✅ Dataset variant generation completed.")

if __name__ == "__main__":
    generate_variants()